# 🎵 Waveform Studio — Audio Visualizer Video Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnchrmXD/waveform-studio/blob/main/waveform_studio.ipynb)

Generate studio-grade, audio-reactive visualizer videos directly in Google Colab using **Waveform Studio** (`mnchrmXD/waveform-studio`).

This notebook is the recommended environment to upload and execute rendering payloads:
1. **Upload or Paste Payload**: Easily upload a `payload.json` file exported from Waveform Studio's **Payload Generator**, or paste the JSON directly.
2. **Custom Visualizer Styles**: Supports Mirrored Bars, Spectrum Bars, Smooth Wave, Radial Spectrum, Digital Matrix, Spine, and Spectrum Bands.
3. **Headless Execution**: Sends payloads to the Waveform Studio headless rendering engine (`/api/render-video`) with real-time frame throughput.
4. **Direct Playback & Export**: In-notebook video preview and direct download of the finished MP4 or transparent alpha WebM.

## 1. Setup & Dependencies
Install `requests` and `tqdm` (for download progress visualization).

In [ ]:
!pip install -q requests tqdm

import requests
import json
import os
import time
from tqdm import tqdm
from IPython.display import HTML, Video, display

## 2. Server Configuration
Specify your Waveform Studio server URL:
- If connecting to a local or cloud-hosted instance (e.g. Google Cloud Run, ngrok, or VPS), enter its base URL.
- If you are running the Waveform Studio Node.js server inside this Colab session, use `http://localhost:3000` (see Section 6).

In [ ]:
# Target Waveform Studio Server URL
API_URL = "http://localhost:3000"

# Check server connectivity and active rendering capabilities
try:
    health_resp = requests.get(f"{API_URL}/api/health", timeout=5)
    if health_resp.status_code == 200:
        print("✅ Connected to Waveform Studio server!")
        print(json.dumps(health_resp.json(), indent=2))
    else:
        print(f"⚠️ Server returned status {health_resp.status_code}: {health_resp.text}")
except Exception as e:
    print(f"⚠️ Could not reach server at {API_URL}.")
    print("If running the backend inside Colab, run Section 6 to start the server.")

## 3. Upload or Configure Payload
You can load your payload into Colab using any of these 3 methods:
- **Method A**: Upload a `payload.json` file exported from Waveform Studio's **Payload Generator** into Colab's file browser (or use the upload button below).
- **Method B**: Paste the JSON payload text directly into `PASTED_JSON`.
- **Method C**: Use the default Python configuration dictionary below.

In [ ]:
# Optional: Run this cell to prompt for uploading a 'payload.json' file from your computer
try:
    from google.colab import files
    print("Upload your payload.json (optional, or skip if configuring below):")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.json'):
            os.rename(fn, 'payload.json')
            print(f"✅ Successfully uploaded and set as payload.json!")
            break
except ImportError:
    pass

In [ ]:
# Paste JSON here if copied from Waveform Studio (leave empty to use Python config or uploaded file)
PASTED_JSON = """"""

payload = None

# 1. Check if payload was pasted
if PASTED_JSON.strip():
    try:
        payload = json.loads(PASTED_JSON)
        print("✅ Loaded payload from PASTED_JSON!")
    except json.JSONDecodeError as err:
        print(f"⚠️ Error parsing PASTED_JSON: {err}")

# 2. Check if payload.json was uploaded
if payload is None and os.path.exists("payload.json"):
    try:
        with open("payload.json", "r") as f:
            payload = json.load(f)
        print("✅ Loaded payload from uploaded payload.json!")
    except Exception as err:
        print(f"⚠️ Error reading payload.json: {err}")

# 3. Fallback to default Python configuration
if payload is None:
    print("ℹ️ Using default Python payload configuration:")
    VIDEO_CONFIG = {
        "width": 1920,       # 1280 (720p), 1920 (1080p Full HD), 3840 (4K UHD)
        "height": 1080,
        "fps": 30,           # 30 or 60 fps
        "format": "mp4"      # "mp4" (H.264/AAC) or "webm" (VP9/Opus, supports transparent alpha)
    }

    SETTINGS = {
        "style": "mirrored-bars",        # "mirrored-bars", "bars-up", "smooth-wave", "radial", "digital-matrix", "spine", "spectrum-bands"
        "barCount": 80,                  # Bar density (16 to 128)
        "heightScale": 1.2,              # Amplitude multiplier (0.2 to 3.0)
        "smoothing": 0.65,               # Temporal FFT smoothing (0.0 to 1.0)
        "sensitivity": 1.0,              # Volume sensitivity gain (0.2 to 3.0)
        "softKneeCompression": True,     # Soft-knee peak compression
        "backgroundType": "dark-studio", # "dark-studio", "oled-black", "light-canvas", "gradient-mesh", "transparent"
        "enableJoint": True,             # Edge & profile tapering
        "jointWidth": 20,                # Taper transition width (5% to 40%)
        "jointCurve": "smooth",          # "smooth", "linear", "cubic"
        "trackTitle": "Midnight Horizons",
        "artistName": "Waveform Studio",
        "showTrackInfo": True,
        "infoPosition": "top-left",
        "showProfileImage": True,
        "profileImageShape": "circle",
        "profileAudioReactiveScale": True
    }

    THEME = "cyber-cyan"  # "cyber-cyan", "electric-indigo", "sunset-ember", "emerald-mint", "monochrome-luxe", "solar-flare", "nordic-frost"
    AUDIO_URL = "https://cdn.freesound.org/previews/612/612627_11861866-lq.mp3"
    PROFILE_IMAGE_URL = "https://images.unsplash.com/photo-1511671782779-c97d3d27a1d4?w=400&q=80"

    payload = {
        "video": VIDEO_CONFIG,
        "settings": SETTINGS,
        "theme": THEME,
        "audio": AUDIO_URL,
        "profileImage": PROFILE_IMAGE_URL
    }

video_format = payload.get("video", {}).get("format", "mp4")
print(f"Payload summary: {payload.get('settings', {}).get('style', 'default')} style, format: {video_format}")

## 4. Render Video via Headless API
Submit the payload to `/api/render-video`. The server analyzes the audio, generates frames via Node.js Canvas, encodes them with FFmpeg (`ultrafast` preset), and streams the video back in real-time.

In [ ]:
OUTPUT_FILENAME = f"waveform_render.{payload.get('video', {}).get('format', 'mp4')}"
render_url = f"{API_URL}/api/render-video"

print(f"🚀 Submitting render request to {render_url}...")
start_time = time.time()

try:
    response = requests.post(render_url, json=payload, stream=True, timeout=300)
    
    if response.status_code == 200:
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024  # 1MB chunks
        
        with open(OUTPUT_FILENAME, 'wb') as f, tqdm(
            desc=OUTPUT_FILENAME,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
                    
        elapsed = time.time() - start_time
        file_size_mb = os.path.getsize(OUTPUT_FILENAME) / (1024 * 1024)
        print(f"\n🎉 Render successfully completed in {elapsed:.1f} seconds!")
        print(f"📁 Output saved to: {OUTPUT_FILENAME} ({file_size_mb:.2f} MB)")
    else:
        print(f"❌ Render failed with HTTP status {response.status_code}: {response.text}")
except Exception as e:
    print(f"❌ Request failed: {e}")

## 5. Preview & Playback Video
Preview the generated video directly within Colab.

In [ ]:
if os.path.exists(OUTPUT_FILENAME):
    print("Previewing rendered video:")
    display(Video(OUTPUT_FILENAME, embed=True, width=720))
else:
    print(f"File {OUTPUT_FILENAME} does not exist yet. Run Section 4 first.")

## 6. (Optional) Run Waveform Studio Server Inside Colab
If you don't have an external server running, you can run Waveform Studio's Node.js + FFmpeg backend directly in this Colab environment.

In [ ]:
# 1. Ensure Node.js (v20+) is available
!node -v || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs)

# 2. Clone repository
# !git clone https://github.com/mnchrmXD/waveform-studio.git
# %cd waveform-studio
# !npm install

# 3. Launch the server in the background
# import subprocess
# server_proc = subprocess.Popen(["npm", "run", "dev"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# time.sleep(6)  # Allow Vite & Express to bind to port 3000
# print("Local Waveform Studio server running on http://localhost:3000!")

## 7. Download Video to Local Machine
Download the finished video to your computer.

In [ ]:
try:
    from google.colab import files
    if os.path.exists(OUTPUT_FILENAME):
        print(f"Downloading {OUTPUT_FILENAME}...")
        files.download(OUTPUT_FILENAME)
    else:
        print(f"File {OUTPUT_FILENAME} not found.")
except ImportError:
    print(f"Not running in Google Colab. File is saved at: {os.path.abspath(OUTPUT_FILENAME)}")